# 🌿 Plant Disease Classification with MobileNetV2 Transfer Learning

This notebook demonstrates transfer learning using **MobileNetV2** (pre-trained on ImageNet) to classify plant diseases across 38 categories from the PlantVillage dataset.

### Advantages of MobileNetV2:
- **Lightweight:** Only ~24 MB compared to ~90 MB for standard CNNs
- **High Accuracy:** Leverages rich ImageNet feature representations
- **Fast Inference:** Optimized for mobile and web deployment


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

print(f'TensorFlow Version: {tf.__version__}')


## 1. Dataset Preprocessing
- Target resolution: 224x224 (native MobileNetV2 input size)
- Batch size: 32


In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_DIR = './plant_disease_dataset'

train_dir = os.path.join(DATASET_DIR, 'train')
valid_dir = os.path.join(DATASET_DIR, 'valid')

# Load image datasets
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

valid_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)


## 2. Building the MobileNetV2 Transfer Learning Architecture

In [ ]:
# Load pre-trained MobileNetV2 backbone (without top classification layer)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model during initial transfer learning
base_model.trainable = False

# Add custom classification head for 38 disease categories
inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(38, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='PlantDisease_MobileNetV2')
model.summary()


## 3. Compiling and Training
Using Adam optimizer with categorical crossentropy.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Initial training (10 epochs)
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=10
)


## 4. Saving the MobileNetV2 Model

In [ ]:
model.save('mobilenet_v2_plant_disease.keras')
print('MobileNetV2 model saved successfully in native .keras format!')
